输入 [("张三", 95), ("李四", 87)]，输出 ['1. 张三: 95', '2. 李四: 87']。不要手动维护序号。

In [3]:
# list里面套tuple 不可变dict
inputs = [("张三", 95), ("李四", 87)]
for i,inputs in enumerate(inputs, start=1):
    print(f"{i}.{inputs}") 

1.('张三', 95)
2.('李四', 87)


输入商品列表，筛选有库存商品，按价格升序、同价时按商品名升序，返回前两个商品名。

products = [
    {"name": "B", "price": 30, "stock": 1},
    {"name": "A", "price": 20, "stock": 0},
    {"name": "D", "price": 20, "stock": 2},
    {"name": "C", "price": 20, "stock": 4},
]
# 预期：['C', 'D']

In [4]:
products = [
    {"name": "B", "price": 30, "stock": 1},
    {"name": "A", "price": 20, "stock": 0},
    {"name": "D", "price": 20, "stock": 2},
    {"name": "C", "price": 20, "stock": 4},
]

def get_score(products :list[dict]) -> list[str]:
    """
    输入商品列表，筛选有库存商品，按价格升序、同价时按商品名升序，返回前两个商品名。
    """
    products_with_stock = [p for p in products if p["stock"] > 0]
    sorted_products = sorted(products_with_stock, key=lambda x: (x["price"], x["name"]))
    return [p["name"] for p in sorted_products[:2]]

get_score(products)

['C', 'D']

题目 3：生成器与惰性处理
实现 even_squares(n)，使用 yield 依次生成 0..n-1 中偶数的平方，而不是一次返回列表。

In [9]:
from collections.abc import Iterator

def even_squares(n:int) -> Iterator[int]:
    for i in range(n):
        if(i % 2 == 0):
            yield i * i
print(list(even_squares(5)))

[0, 4, 16]


为任意同步函数编写 timed 装饰器，保留函数元数据，返回值不能改变。

In [10]:
def timed(func):
    def wrapper(*args, **kw):
        print('ok')
        return func(*args, **kw)
    return wrapper

@timed
def add(x: int, y: int) -> int:
    return x + y

print(add(1, 2))

ok
3


In [ ]:
from functools import wraps          # 保留被装饰函数的元数据（名字、文档等）
from time import perf_counter        # 高精度计时器
from collections.abc import Callable # 类型提示：可调用对象
from typing import ParamSpec, TypeVar # 泛型工具，用于精确标注装饰器类型


P = ParamSpec("P")  # 捕获函数的参数签名（positional + keyword）
R = TypeVar("R")    # 捕获函数的返回类型


def timed(func: Callable[P, R]) -> Callable[P, R]:
    @wraps(func)                          # 把 func 的 __name__、__doc__ 等复制给 wrapper
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start = perf_counter()            # 记录开始时间
        try:
            return func(*args, **kwargs)  # 调用原函数
        finally:
            # finally 无论是否抛出异常都会执行:.4f 格式化为 4 位小数的浮点数
            print(f"{func.__name__}: {perf_counter()-start:.4f}s")
    return wrapper

@timed
def add(a: int, b: int) -> int:
    return a + b

assert add(2, 3) == 5
assert add.__name__ == "add"

add: 0.0000s


使用 contextlib.contextmanager 写 temporary_value(config, key, value)：在 with 块中临时替换字典值，退出后恢复；不存在的键应在退出时删除。

In [ ]:
from contextlib import contextmanager
from collections.abc import Iterator

# 用来区分「key 不存在」和「key 存在但值为 None」这两种情况如果 
_SENTINEL = object()

'''@contextmanager 是标准库 contextlib 提供的装饰器，
作用是把一个生成器函数改造成可以直接用在 with 语句里的上下文管理器。
它内部会自动给这个函数套上 _GeneratorContextManager 类（这个类实现了 __enter__ / __exit__）。
'''
@contextmanager
def temporary_value(config: dict, key: str, value: object) -> Iterator[None]:
    # dict.get 返回 _SENTINEL，说明原来没有这个 key；否则说明有这个 key。用 None 做默认值就不行了，因为字典里可能本来就存着 None。
    old_value = config.get(key, _SENTINEL)
    config[key] = value
    try:
        yield
    finally:
        if old_value is _SENTINEL:
            #dict.pop(key, None) —— 带默认值删除
            config.pop(key, None)
        else:
            config[key] = old_value

cfg = {"timeout": 5}
with temporary_value(cfg, "timeout", 20):
    assert cfg["timeout"] == 20
assert cfg == {"timeout": 5}